In [ ]:
import json
import re

def clean_value(val):
    """Làm sạch các tiền tố và ký tự dư thừa sau khi trích xuất"""
    if val:
        # Xóa các tiền tố phổ biến nếu Regex còn sót lại
        val = re.sub(r'^(?:chữ Hán|Hán tự|tên húy là|tên thật là|húy là|tên thật|tên húy):\s*', '', val, flags=re.IGNORECASE)
        return val.strip().strip(',').strip(';').strip('.').strip()
    return "Không rõ"

def extract_extra_info(summary):
    """Sử dụng Regex để trích xuất thông tin từ đoạn tóm tắt (Summary)"""
    info = {
        "chu_han_extracted": "Không rõ",
        "ten_huy_that_extracted": "Không rõ",
        "ngay_nam_sinh_mat_extracted": "Không rõ",
        "que_quan_extracted": "Không rõ"
    }
    
    if not summary:
        return info

    # 1. Trích xuất Chữ Hán
    # Tìm trong ngoặc đơn hoặc sau cụm 'chữ Hán:'
    han_match = re.search(r'(?:chữ Hán|Hán tự):\s*([\u4e00-\u9fff\s,]+)', summary)
    if not han_match:
        # Dự phòng tìm bất kỳ cụm chữ Hán nào trong ngoặc đơn đầu tiên
        han_match = re.search(r'\((?:[^)]*?)([\u4e00-\u9fff]{2,})', summary)
    
    if han_match:
        info["chu_han_extracted"] = clean_value(han_match.group(1))

    # 2. Trích xuất Tên húy / Tên thật
    huy_match = re.search(r'(?:tên húy là|tên thật là|húy là|tên thật|tên húy)\s+([^,;()\n.]{1,30})', summary, re.IGNORECASE)
    if huy_match:
        info["ten_huy_that_extracted"] = clean_value(huy_match.group(1))
    
    # 3. Trích xuất Ngày/Năm sinh và mất
    # Tìm các dải năm hoặc ngày tháng trong ngoặc đơn (Ví dụ: 1023 - 1072 hoặc 30 tháng 3... )
    life_pattern = r'(\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}|\d{1,4}(?:\s*TCN)?)\s*[–-]\s*(\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}|\d{1,4}(?:\s*TCN)?)'
    parens = re.findall(r'\(([^)]+)\)', summary)
    for p in parens:
        if "trị vì" in p.lower(): continue # Bỏ qua dải năm trị vì
        life_match = re.search(life_pattern, p)
        if life_match:
            info["ngay_nam_sinh_mat_extracted"] = life_match.group(0).strip()
            break
            
    # 4. Trích xuất Nơi sinh / Quê quán
    place_patterns = [
        r'người\s+(?:huyện|xã|tỉnh|vùng|quê ở)\s+([^,;.\n]+)',
        r'quê ở\s+([^,;.\n]+)',
        r'sinh tại\s+([^,;.\n]+)'
    ]
    for pattern in place_patterns:
        pm = re.search(pattern, summary, re.IGNORECASE)
        if pm:
            info["que_quan_extracted"] = clean_value(pm.group(1))
            break

    return info

def enrich_json_data(input_file, output_file):
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        enriched_data = []
        for item in data:
            # Lấy summary để trích xuất
            summary = item.get('summary', '')
            
            # Trích xuất thông tin mới
            extra_info = extract_extra_info(summary)
            
            # Kết hợp dữ liệu cũ (item) với thông tin mới (extra_info)
            # Chúng ta sẽ đưa các trường mới lên đầu để dễ quan sát
            combined_item = {
                "name": item.get("name"),
                "matched_name": item.get("matched_name"),
                **extra_info, # Bung các trường mới vào đây
                "summary": item.get("summary"),
                "content_hierarchy": item.get("content_hierarchy"),
                "citations": item.get("citations"),
                "url": item.get("url")
            }
            enriched_data.append(combined_item)
            
        # Lưu ra file mới
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(enriched_data, f, ensure_ascii=False, indent=4)
            
        print(f"Xử lý hoàn tất! Đã cập nhật {len(enriched_data)} nhân vật.")
        print(f"Kết quả lưu tại: {output_file}")

    except Exception as e:
        print(f"Đã xảy ra lỗi: {e}")

if __name__ == "__main__":
    # Đảm bảo bạn đã có file này trong thư mục
    enrich_json_data("data_lich_su_chi_tiet.json", "data_lich_su_nang_cao.json")

Xử lý hoàn tất! Đã cập nhật 71 nhân vật.
Kết quả lưu tại: data_lich_su_nang_cao.json


: 